# Entrenamiento de modelo de predicción de categorías


## Importación de librerias necesarias


In [2]:
%load_ext IPython.extensions.autoreload
%autoreload 2

In [3]:
import os
from pathlib import Path
import sys

def find_src_folder(current_path: Path, folder_name: str = 'src') -> Path:
    search_directories = [current_path] + list(current_path.parents)
    for parent in search_directories:
        if parent.name == folder_name:
            return parent.parent
    return current_path

src_path = find_src_folder(Path.cwd(), 'src')
sys.path.append(str(src_path))

print(f"JAVA_HOME: {os.environ.get('JAVA_HOME')}")
print(f"TFHUB_CACHE_DIR: {os.environ.get('TFHUB_CACHE_DIR')}")


JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64
TFHUB_CACHE_DIR: /mnt/d/Maestría/Amazon Reviews Code/tf_cache


In [4]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


2025-10-17 21:26:28.501304: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-17 21:26:28.531091: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-17 21:26:28.531652: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-17 21:26:30.440910: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2025-10-17 21:26:34.272099: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Do

[]


In [5]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F, types as T, DataFrame
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from pyspark.ml.functions import array_to_vector, vector_to_array
from pyspark.ml.feature import VectorAssembler

In [6]:
from src.utils.spark import SparkUtils, METASTORE_PATH
spark_utils = SparkUtils('predict_category_model')
spark = spark_utils.spark


:: loading settings :: url = jar:file:/mnt/d/Maestr%c3%ada/Amazon%20Reviews%20Code/.venv-linux/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/edgar/.ivy2/cache
The jars for the packages stored in: /home/edgar/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bea3f960-fc33-4f22-81bb-35580c4f4da4;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 235ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   

## Cargar datos fuente de entrenamiento

In [7]:
GOLD_ENCODING = 'gold.encoding'
GOLD_PREMODELING = 'gold.premodeling'
GOLD_SCHEMA_CLUSTER = 'gold.cluster'
SCHEMA = 'silver.preprocess'

In [8]:
METASTORE_PATH

'/mnt/d/Maestría/Amazon Reviews Code/data'

In [25]:
import os
import shutil

src_dir = f"{METASTORE_PATH}/warehouse/gold.premodeling/final_training_data_rating_array"
dst_dir = f"{METASTORE_PATH}/warehouse/gold.premodeling/dataset_score_model"
dst_path = os.path.join(dst_dir, "dataset_training.parquet")

os.makedirs(dst_dir, exist_ok=True)

parquet_files = [f for f in os.listdir(src_dir) if f.endswith(".parquet")]
print(parquet_files)
if len(parquet_files) == 0:
    raise FileNotFoundError(f"No parquet files found in {src_dir}")
elif len(parquet_files) > 1:
    raise RuntimeError(f"Expected exactly one parquet file in {src_dir}, found {len(parquet_files)}")

src_path = os.path.join(src_dir, parquet_files[0])

shutil.copy(src_path, dst_path)

['part-00000-1c96f0b0-f2b8-4013-8997-b2983e128201-c000.snappy.parquet']


'/mnt/d/Maestría/Amazon Reviews Code/data/warehouse/gold.premodeling/dataset_score_model/dataset_training.parquet'

In [42]:
import tensorflow as tf
import tensorflow_io as tfio

label_col = b"rating"  # note the 'b' prefix!

dataset = tfio.IODataset.from_parquet( f"/mnt/d/Maestría/Amazon Reviews Code/data/warehouse/gold.premodeling/final_training_data_rating_array/part-00000-1c96f0b0-f2b8-4013-8997-b2983e128201-c000.snappy.parquet" )

def split_xy(row):
    y = row[label_col]
    feature_cols = [k for k in row.keys() if k != label_col]
    x = tf.stack([tf.cast(row[k], tf.float32) for k in feature_cols], axis=-1)
    return x, y

dataset = (
    dataset.map(split_xy)
            .shuffle(10000)
            .batch(128)
            .prefetch(tf.data.AUTOTUNE)
)


In [43]:
def create_simple_nn(input_shape, num_classes=5):
    """Create a simple neural network for rating prediction"""
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_shape,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [45]:
sample_batch = next(iter(dataset.take(1)))
X_sample, y_sample = sample_batch
input_shape = X_sample.shape[1]

print(f"✓ Dataset loaded successfully")
print(f"Input shape: {input_shape}")
print(f"Sample X shape: {X_sample.shape}")
print(f"Sample y shape: {y_sample.shape}")

# Create model
model = create_simple_nn(input_shape=input_shape, num_classes=6)

print("\nModel Architecture:")
model.summary()

# Split dataset for training and validation
# Take a reasonable subset for training to avoid memory issues
train_size = 800  # Adjust based on your memory
val_size = 200

train_dataset = dataset.take(train_size)
val_dataset = dataset.skip(train_size).take(val_size)

print(f"\nDataset split:")
print(f"Training samples: {train_size}")
print(f"Validation samples: {val_size}")

# Train the model
print("\n🚀 Starting training...")

# Add callbacks for better training
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

2025-10-17 22:38:18.365112: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_14234' with dtype int64 and shape [1]
	 [[{{node Placeholder/_14234}}]]
2025-10-17 22:38:18.456024: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_7952' with dtype int64 and shape [1]
	 [[{{node Placeholder/_7952}}]]
2025-10-17 22:38:51.882987: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 9617 of 10000
2025-10-17 22:38:52.114882: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:417] Shuffle buffer filled.


✓ Dataset loaded successfully
Input shape: 1025
Sample X shape: (128, 1025)
Sample y shape: (128,)

Model Architecture:
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_4 (Dense)             (None, 128)               131328    
                                                                 
 dropout_2 (Dropout)         (None, 128)               0         
                                                                 
 dense_5 (Dense)             (None, 64)                8256      
                                                                 
 dropout_3 (Dropout)         (None, 64)                0         
                                                                 
 dense_6 (Dense)             (None, 32)                2080      
                                                                 
 dense_7 (Dense)             (None, 6)                 198       


2025-10-17 22:38:54.598689: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_6548' with dtype int64 and shape [1]
	 [[{{node Placeholder/_6548}}]]
2025-10-17 22:38:54.683275: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_12052' with dtype int64 and shape [1]
	 [[{{node Placeholder/_12052}}]]


Epoch 1/10
    800/Unknown - 80s 87ms/step - loss: 1.1819 - accuracy: 0.5522

2025-10-17 22:40:38.732269: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_7412' with dtype int64 and shape [1]
	 [[{{node Placeholder/_7412}}]]
2025-10-17 22:40:38.823864: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_9910' with dtype int64 and shape [1]
	 [[{{node Placeholder/_9910}}]]
2025-10-17 22:41:14.498107: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 7254 of 10000
2025-10-17 22:41:16.811925: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:417] Shuffle buffer filled.


...

800/800 [==============================] - 200s 238ms/step - loss: 0.7335 - accuracy: 0.7329 - val_loss: 1.5585 - val_accuracy: 0.4845 - lr: 0.0010
Epoch 3/10
800/800 [==============================] - ETA: 0s - loss: 0.4866 - accuracy: 0.8263

2025-10-17 22:47:53.416107: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 7341 of 10000
2025-10-17 22:47:55.636490: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:417] Shuffle buffer filled.



Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
800/800 [==============================] - 209s 249ms/step - loss: 0.4866 - accuracy: 0.8263 - val_loss: 1.7479 - val_accuracy: 0.4958 - lr: 0.0010
Epoch 4/10
800/800 [==============================] - ETA: 0s - loss: 0.3368 - accuracy: 0.8816

2025-10-17 22:51:22.743408: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 6935 of 10000
2025-10-17 22:51:25.200484: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:417] Shuffle buffer filled.


Restoring model weights from the end of the best epoch: 1.
800/800 [==============================] - 205s 244ms/step - loss: 0.3368 - accuracy: 0.8816 - val_loss: 1.8738 - val_accuracy: 0.4934 - lr: 5.0000e-04
Epoch 4: early stopping


In [37]:
next(iter(dataset.take(10)))

2025-10-17 22:20:13.060709: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_10' with dtype int64 and shape [1]
	 [[{{node Placeholder/_10}}]]
2025-10-17 22:20:13.061096: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_46' with dtype int64 and shape [1]
	 [[{{node Placeholder/_46}}]]


OrderedDict([(b'rating', <tf.Tensor: shape=(), dtype=float32, numpy=5.0>),
             (b'helpful_vote', <tf.Tensor: shape=(), dtype=int32, numpy=0>),
             (b'combined_features.list.element',
              <tf.Tensor: shape=(), dtype=float64, numpy=-0.010146216547582299>)])